# AJOU-VIS Qwen3-TTS 한국어 벤치마크

1~2주차 이월 과제: Qwen3-TTS를 **양자화 수준별(GGML BF16 / Q8_0 / Q4_K_M)**로
**한국어 음질(청취 평가) / 첫 오디오까지 지연(TTFC) / RTF / GPU 메모리** 기준으로 비교한다.

> 참고: 계획서의 "4/6/8bit"는 Mac 전용 MLX 백엔드 옵션이고, CUDA(GGML) 백엔드의 양자화는
> BF16(원본)·Q8_0(8bit급)·Q4_K_M(4bit급)이다. 서버 GPU에서 서빙하므로 GGML 기준으로 비교한다.

**사용법**
1. 런타임 → GPU 선택 (T4/A100 무관, 비교표에 GPU 자동 기록)
2. 위에서부터 순서대로 실행 — 업로드할 파일 없음 (합성할 문장은 3번 셀에 정의)
3. 5번 셀에서 양자화별 합성 음성을 직접 듣고 평가
4. 마지막 셀에서 비교표 CSV + 청취용 오디오 zip 다운로드 → 팀 Drive에 공유해 전원 청취 평가

**크래시 대비 구조**: GGML이 간혹 네이티브 크래시(segfault)를 내므로, 4번 셀은 양자화마다
**별도 프로세스**를 띄워 실행한다 — 크래시가 나도 커널은 살아남아 실패를 기록하고 다음
양자화로 넘어간다. 결과는 양자화별로 `tts_results/`에 즉시 저장되므로, 만에 하나 커널이
재시작되더라도 2번 → 3번 → 4번 셀만 다시 실행하면 완료된 양자화는 건너뛰고 이어서 진행된다.

> 오디오 파일(.wav)은 레포에 커밋하지 않는다(.gitignore 규칙) — Drive에 공유할 것.

In [ ]:
# 1. GPU 확인
!nvidia-smi

In [ ]:
# 2. 포크 레포 클론 + 의존성 설치 (몇 분 걸림) — 재실행해도 안전 (커널 재시작 후 여기부터)
import os
if not os.path.isdir("/content/speech-to-speech"):
    !git clone --depth 1 https://github.com/AJOU-VIS/speech-to-speech.git /content/speech-to-speech
%cd /content/speech-to-speech
!pip install -q .
!pip install -q soundfile pandas
# 다음 셀에서 import 에러가 나는 경우에만 런타임 재시작 후 이 셀부터 다시 실행.

In [ ]:
# 3. 합성할 한국어 문장 — 치토가 실제로 말할 답변 형태로 구성
# (짧은 안내 / 근거 인용 / 숫자·기한 / 보류 응답 / 영어 혼용 대비 영어 1문장)
SENTENCES = {
    "ko_greeting": "안녕하세요, 저는 아주대 안내 가이드 치토예요! 학사, 행정, 캠퍼스 뭐든 물어봐요.",
    "ko_citation": "학사요람 졸업요건에 따르면 소프트웨어학과는 2024학번 기준 졸업까지 140학점이 필요해요.",
    "ko_numbers": "수강신청 정정 기간은 개강 후 첫 주이고, 최대 19학점까지 신청할 수 있어요.",
    "ko_deferral": "음, 그건 제가 가진 자료로는 확실하지 않아요. 정확한 건 학사지원팀에 확인해보는 게 좋아요!",
    "en_sample": "According to the Academic Handbook, 2024-entry Software students need 140 credits to graduate.",
}

In [ ]:
# 4. 벤치마크 실행 — 양자화별로 "별도 프로세스"에서 모델을 로드·합성하고 wav로 저장
# 측정: 모델 로드 시간, 문장별 합성 시간, 첫 오디오 청크까지 지연(TTFC), RTF, GPU 메모리 증가량
#
# ★ 크래시 격리: GGML이 네이티브 크래시(segfault)를 내면 같은 프로세스인 커널까지 통째로
#   죽는다. 그래서 양자화 하나마다 자식 프로세스(bench_one_tts.py)를 띄워 실행한다 —
#   크래시가 나도 자식만 죽고, 커널은 실패를 기록한 뒤 다음 양자화로 넘어간다.
#   결과는 tts_results/<양자화>.json 에 저장되며, 이미 있는 양자화는 재실행 시 건너뛴다.
import json, subprocess, sys
from pathlib import Path

import numpy as np
import torch

GPU_NAME = torch.cuda.get_device_name(0)
RESULT_DIR = Path("tts_results"); RESULT_DIR.mkdir(exist_ok=True)
Path("sentences.json").write_text(json.dumps(SENTENCES, ensure_ascii=False))
print("측정 GPU:", GPU_NAME)

# BF16(원본) → Q8_0(8bit급) → Q4_K_M(4bit급) 순서로 음질 저하 vs 속도·메모리 이득을 확인
QUANTIZATIONS = ["BF16", "Q8_0", "Q4_K_M"]

# 자식 프로세스용 스크립트. 결과를 cleanup 전에 저장하고, 메모리 해제는 프로세스 종료에 맡긴다
# (cleanup 중 double-free 크래시 가능성도 피하기 위해 명시적 cleanup을 하지 않음).
Path("bench_one_tts.py").write_text('''
import json, sys, time
from pathlib import Path
from queue import Queue
from threading import Event

import numpy as np
import soundfile as sf
import torch

from speech_to_speech.pipeline.messages import TTSInput
from speech_to_speech.TTS.qwen3_tts_handler import Qwen3TTSHandler

quant = sys.argv[1]
SENTENCES = json.loads(Path("sentences.json").read_text())
SAMPLE_RATE = 16000
OUT_DIR = Path("tts_samples"); OUT_DIR.mkdir(exist_ok=True)
RESULT_DIR = Path("tts_results"); RESULT_DIR.mkdir(exist_ok=True)


def gpu_used_mb():
    free, total = torch.cuda.mem_get_info()
    return (total - free) / 1024**2


def synthesize(handler, text, lang):
    start = time.perf_counter()
    chunks, ttfc = [], None
    for chunk in handler.process(TTSInput(text=text, language_code=lang)):
        if chunk is None:
            continue
        if ttfc is None:
            ttfc = time.perf_counter() - start
        chunks.append(np.asarray(chunk))
    elapsed = time.perf_counter() - start
    audio = np.concatenate(chunks) if chunks else np.zeros(0, dtype=np.int16)
    return audio, elapsed, ttfc


mem_before = gpu_used_mb()
t0 = time.perf_counter()
handler = Qwen3TTSHandler(
    Event(), queue_in=Queue(), queue_out=Queue(), setup_args=(Event(),),
    setup_kwargs={"device": "cuda", "backend": "ggml",
                  "ggml_quantization": quant, "language": "auto"})
load_time = time.perf_counter() - t0

first_key = next(iter(SENTENCES))
synthesize(handler, SENTENCES[first_key], "ko")  # 실측 제외용 워밍업 1회

per_sentence = []
for key, text in SENTENCES.items():
    lang = "en" if key.startswith("en") else "ko"
    audio, elapsed, ttfc = synthesize(handler, text, lang)
    duration = len(audio) / SAMPLE_RATE
    wav_path = OUT_DIR / f"{quant}_{key}.wav"
    sf.write(wav_path, audio, SAMPLE_RATE)
    per_sentence.append({"key": key, "seconds": elapsed, "ttfc": ttfc,
                         "audio_seconds": duration, "wav": str(wav_path)})
    print(f"  {key}: 합성 {elapsed:.2f}s, TTFC {ttfc:.2f}s, 오디오 {duration:.1f}s", flush=True)

(RESULT_DIR / f"{quant}.json").write_text(json.dumps(
    {"quant": quant, "status": "ok", "load_time_s": load_time,
     "gpu_mem_mb": gpu_used_mb() - mem_before,
     "gpu": torch.cuda.get_device_name(0), "sentences": per_sentence},
    ensure_ascii=False, indent=2))
print(f"  → 결과 저장: tts_results/{quant}.json", flush=True)
''')

for quant in QUANTIZATIONS:
    result_path = RESULT_DIR / f"{quant}.json"
    if result_path.exists():
        print(f"===== {quant} ===== 이미 완료됨, 건너뜀")
        continue
    print(f"\n===== {quant} =====")
    proc = subprocess.run([sys.executable, "bench_one_tts.py", quant])
    if proc.returncode != 0 and not result_path.exists():
        print(f"  프로세스 비정상 종료 (code {proc.returncode}) — 실패로 기록하고 계속 진행")
        result_path.write_text(json.dumps(
            {"quant": quant, "status": "failed",
             "error": f"process exited with code {proc.returncode} (native crash)"},
            ensure_ascii=False))

results = [json.loads(p.read_text()) for p in RESULT_DIR.glob("*.json")]
results.sort(key=lambda r: QUANTIZATIONS.index(r["quant"]) if r["quant"] in QUANTIZATIONS else 99)
with open("tts_benchmark_raw.json", "w", encoding="utf-8") as f:
    json.dump({"gpu": GPU_NAME, "sentences": SENTENCES, "results": results}, f,
              ensure_ascii=False, indent=2)
print(f"\n완료: { {r['quant']: r['status'] for r in results} }")

In [ ]:
# 5. 청취 평가 — 양자화별 합성 결과를 문장 단위로 비교해서 듣기
# 특히 확인할 것: 한국어 발음 명료도, 숫자 읽기(140학점, 19학점), 어색한 억양·잡음,
# 그리고 Q4_K_M에서 음질 열화가 체감되는지 (열화가 없으면 메모리 이득이 커서 Q4 채택 유력)
from IPython.display import Audio, display

for key in SENTENCES:
    print(f"\n────── {key}: {SENTENCES[key]}")
    for r in results:
        if r["status"] != "ok":
            continue
        entry = next(s for s in r["sentences"] if s["key"] == key)
        print(f"  [{r['quant']}]")
        display(Audio(entry["wav"]))

In [ ]:
# 6. 비교표
import pandas as pd

rows = []
for r in results:
    if r["status"] != "ok":
        rows.append({"양자화": r["quant"], "상태": "실패"})
        continue
    ss = r["sentences"]
    rows.append({
        "양자화": r["quant"],
        "GPU": GPU_NAME,
        "상태": "ok",
        "평균 합성(s)": round(float(np.mean([s["seconds"] for s in ss])), 3),
        "평균 TTFC(s)": round(float(np.mean([s["ttfc"] for s in ss])), 3),
        "RTF": round(float(np.mean([s["audio_seconds"] / s["seconds"] for s in ss])), 2),
        "모델 로드(s)": round(r["load_time_s"], 1),
        "GPU 메모리(MB)": round(r["gpu_mem_mb"]),
        "청취 평가": "",  # 팀 청취 후 기입 (예: 이상 없음 / 숫자 발음 뭉개짐)
    })

summary = pd.DataFrame(rows)
summary.to_csv("tts_benchmark_summary.csv", index=False)
print("=== 요약 (TTS는 RTF > 1 이면 실시간보다 빠름) ===")
display(summary)

In [ ]:
# 7. 결과 다운로드 — CSV/JSON은 레포 bench/results/에 커밋, 오디오 zip은 팀 Drive에 공유
import shutil
from google.colab import files

shutil.make_archive("tts_samples", "zip", "tts_samples")
files.download("tts_benchmark_summary.csv")
files.download("tts_benchmark_raw.json")
files.download("tts_samples.zip")